In [1]:
# ============================================
# NOTEBOOK 2 - ANÁLISE DE CENÁRIOS E SENSIBILIDADE
# ============================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from textwrap import fill

# 1️⃣ SIMULAÇÃO DE CENÁRIOS
# --------------------------
# Criaremos três cenários para comparação:
#  - Pessimista: baixo crescimento, churn alto
#  - Realista: cenário base
#  - Otimista: alto crescimento, churn baixo

meses = pd.date_range("2025-11", periods=36, freq="MS")

def simular_cenario(nome, crescimento_trafego, churn, custo_ia):
    base_trafego = 1000
    mrr, caixa = [], []
    saldo = 0
    clientes = 50
    arpu = 97
    for m in range(len(meses)):
        base_trafego *= (1 + crescimento_trafego)
        novos = int(base_trafego * 0.05 * 0.15)
        clientes = clientes + novos - int(clientes * churn)
        receita = clientes * arpu
        custo = clientes * custo_ia
        lucro = receita - custo - 15000  # Opex fixo
        saldo += lucro
        mrr.append(receita)
        caixa.append(saldo)
    return pd.DataFrame({
        "mes": meses,
        "MRR": mrr,
        "Caixa": caixa,
        "cenário": nome
    })

cenarios = pd.concat([
    simular_cenario("Pessimista", 0.05, 0.06, 6),
    simular_cenario("Realista", 0.10, 0.04, 5),
    simular_cenario("Otimista", 0.20, 0.03, 4)
])

# 2️⃣ VISUALIZAÇÃO - GRÁFICO DE CENÁRIOS
# --------------------------------------
fig = go.Figure()

for nome, df in cenarios.groupby("cenário"):
    fig.add_trace(go.Scatter(
        x=df["mes"], y=df["Caixa"],
        mode="lines", name=f"{nome} — Caixa (R$)",
    ))

fig.update_layout(
    title="Projeção de Caixa por Cenário (36 meses)",
    xaxis_title="Mês",
    yaxis_title="Saldo de Caixa (R$)",
    legend_title="Cenários",
    hovermode="x unified"
)
fig.show()

# 📘 LEGENDA EXPLICATIVA:
print(fill("""
💡 Este gráfico mostra como o saldo de caixa evolui sob diferentes condições de crescimento.
O eixo X representa o tempo (36 meses) e o eixo Y o saldo acumulado em reais.
• Cenário Pessimista → Crescimento lento, alto churn (perda de clientes).
• Cenário Realista → Parâmetros de base do modelo.
• Cenário Otimista → Crescimento rápido e retenção alta.
""", width=100))

# 3️⃣ ANÁLISE DE SENSIBILIDADE COM GLossário
# ------------------------------------------
variaveis = {
    "crescimento_trafego": [0.05, 0.1, 0.2],
    "churn_mensal": [0.03, 0.04, 0.05, 0.06],
    "custo_ia": [4, 5, 6, 8]
}

resultados = []
for v, valores in variaveis.items():
    for val in valores:
        df = simular_cenario(v, val, 0.04 if v != "churn_mensal" else val, 5 if v != "custo_ia" else val)
        saldo_final = df["Caixa"].iloc[-1]
        resultados.append({"variavel": v, "valor": val, "saldo_final": saldo_final})

sens = pd.DataFrame(resultados)
pivot = sens.pivot(index="variavel", columns="valor", values="saldo_final")

import plotly.express as px
fig2 = px.imshow(
    pivot,
    color_continuous_scale="RdBu",
    aspect="auto",
    title="Mapa de Calor: Sensibilidade do Caixa ao final do período",
    labels=dict(x="Valor testado", y="Variável", color="Saldo Final (R$)")
)
fig2.show()

# 📘 LEGENDA EXPLICATIVA:
print(fill("""
💡 Este mapa de calor mostra como o saldo de caixa muda conforme ajustamos variáveis-chave:
Eixo X → valor testado para cada variável.
Eixo Y → variável analisada.
Cores quentes (vermelho) indicam melhor resultado financeiro; cores frias (azul) indicam pior.
""", width=100))

# 4️⃣ RESUMO DE INSIGHTS
# -----------------------
insights = {
    "Maior impacto": "A variável 'crescimento_trafego' tem o maior efeito no caixa final.",
    "Sensibilidade média": "O churn e o custo por usuário impactam menos, mas ainda influenciam o ponto de equilíbrio.",
    "Decisão": "Priorizar investimento em aquisição e retenção. Revisar custos de IA apenas após validar crescimento."
}

print("\nResumo de Insights Estratégicos:\n")
for k, v in insights.items():
    print(f"🔹 {k}: {v}")


 💡 Este gráfico mostra como o saldo de caixa evolui sob diferentes condições de crescimento. O eixo
X representa o tempo (36 meses) e o eixo Y o saldo acumulado em reais. • Cenário Pessimista →
Crescimento lento, alto churn (perda de clientes). • Cenário Realista → Parâmetros de base do
modelo. • Cenário Otimista → Crescimento rápido e retenção alta.


 💡 Este mapa de calor mostra como o saldo de caixa muda conforme ajustamos variáveis-chave: Eixo X →
valor testado para cada variável. Eixo Y → variável analisada. Cores quentes (vermelho) indicam
melhor resultado financeiro; cores frias (azul) indicam pior.

Resumo de Insights Estratégicos:

🔹 Maior impacto: A variável 'crescimento_trafego' tem o maior efeito no caixa final.
🔹 Sensibilidade média: O churn e o custo por usuário impactam menos, mas ainda influenciam o ponto de equilíbrio.
🔹 Decisão: Priorizar investimento em aquisição e retenção. Revisar custos de IA apenas após validar crescimento.
